# 1. Prep (one-time)

- Set the inputs 
    -   Bam files: Tumor and normal (optional but recommended to load side-by-side in IGV) files 
    -   Reference file: Human genome Hg38
    -   Breaks from CNVkit

In [ ]:
# set your inputs
BAM_T=WIAB_IDPE_P53_merged_sorted.rg.bam
BAM_N=P53_control_dedup.bam
REF=reference.fasta
BREAKS=sample.breaks
SAMPLE=case01
PAD=500                   # window around breakpoints for IGV zooms

# ensure indices exist
samtools index $BAM_T
samtools index $BAM_N


# 2. Measure insert size to define “abnormal”

- flag discordant pairs by distance/orientation (Same chromossome). 
- Generate library’s normal insert size to know what’s “too long”.
- `https://gatk.broadinstitute.org/hc/en-us/articles/360037055772-CollectInsertSizeMetrics-Picard`
- How to use: take the median insert size (μ) and define a threshold like MAX_IS = μ + 3×SD. Use this to catch same-chromosome, too-far pairs.

In [ ]:
# CollectInsertSizeMetrics Picard tools
picard CollectInsertSizeMetrics I=$BAM_T O=${SAMPLE}.insert.txt H=${SAMPLE}.insert.pdf M=0.5

| Metric                  | Value                 | Meaning                                                                       |
| :---------------------- | :-------------------- | :---------------------------------------------------------------------------- |
| **Median insert size**  | **198 bp**            | midpoint of your fragment-length distribution — about right for exome capture |
| **Mean insert size**    | **210 bp ± 75 bp SD** | average and dispersion                                                        |
| **Mode**                | 178 bp                | most common fragment length                                                   |
| **Minimum / Maximum**   | 2 – 223 M bp          | the extreme max is an artefact (one or a few broken pairs)                    |
| **Read pairs analysed** | 231 M                 | number of FR-oriented pairs used                                              |
| **Orientation**         | FR                    | forward–reverse, normal for Illumina libraries                                |


- Anything > median + 6×SD ≈ 210 + 450 ≈ 660 bp should be treated as “abnormally long.”
- Expect the bulk of normal FR pairs to cluster within 120–300 bp, so in IGV you can color by insert size and immediately spot the few hundred-bp outliers

In [ ]:
# Set max insert size - Used in downstream analysis
MAX_IS=700

# 3. Extract the read evidence sets

- Make three BAMs: 
    -   (a) **inter-chromosomal pairs** - strong hints for translocations/fusions—mates land on different chromosomes. # Every read in your 30 GB BAM must be scanned for an SA:Z tag (the supplementary alignment tag from split alignments)
    -   (b) **long same-chr pairs** - pairs on the same chromosome but separated much further than expected often straddle deletions/duplications/inversions.
    -   (c) **split reads** - hard evidence of the breakpoint—a single read aligns in two chunks.
    
- These are your “evidence tracks” for IGV.

In [ ]:
# 2a - Inter-chromosomal pairs
MAX_IS=700

# (a) Split-reads (supplementary alignments)
samtools view -h WIAB_IDPE_P53_merged_sorted.rg.bam \
  | awk '{if($0 ~ /^@/ || $0 ~ /SA:Z/) print $0}' \
  | samtools view -b -o P53_SETD2i_merged.splitreads.bam
samtools index P53_SETD2i_merged.splitreads.bam


# (b) Inter-chromosomal pairs
samtools view -h WIAB_IDPE_P53_merged_sorted.rg.bam | \
awk '$7!="="' | samtools view -b -o P53_SETD2i_merged.interchr.bam
samtools index P53_SETD2i_merged.interchr.bam

# (c) Discordant / long-insert pairs
samtools view -b -f 1 -F 2 WIAB_IDPE_P53_merged_sorted.rg.bam > P53_SETD2i_merged.longinsert.bam
samtools index P53_SETD2i_merged.longinsert.bam


| File                               | Meaning                                                                                      | Corresponding Notebook Step         |
| :--------------------------------- | :------------------------------------------------------------------------------------------- | :---------------------------------- |
| `P53_SETD2i_merged.longinsert.bam` | Read pairs with unusually long inserts (>700 bp) — potential intra-chromosomal events        | Step 4 (discordant-pair extraction) |
| `P53_SETD2i_merged.interchr.bam`   | Read pairs mapping to **different chromosomes** — potential *translocations/fusions*         | Step 4 (inter-chromosomal subset)   |
| `P53_SETD2i_merged.splitreads.bam` | Reads with **supplementary alignments (SA:Z)** — true split-read evidence across breakpoints | Step 5 (split-read inspection)      |
